In [1]:
import speech_recognition as sr
import sounddevice as sd
from scipy.io.wavfile import write
import whisper
import vosk
import json
import wave
import os
import logging

# Suppress whisper's verbose output
logging.getLogger("whisper").setLevel(logging.ERROR)


# --- Configuration ---
RECORDING_FILENAME = "output.wav"
SAMPLE_RATE = 16000  # 16kHz is standard for STT
VOSK_MODEL_PATH = "vosk-model-small-en-us-0.15" # ⚠️ CHANGE THIS IF YOU DOWNLOADED A DIFFERENT MODEL

print("✅ Imports and configuration loaded.")

✅ Imports and configuration loaded.


In [2]:
def record_audio(filename, duration=5):
    """Records audio from the microphone for a given duration and saves it."""
    print(f"🎤 Speak something for {duration} seconds...")
    try:
        recording = sd.rec(int(duration * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='int16')
        sd.wait()
        write(filename, SAMPLE_RATE, recording)
        print(f"👍 Audio recorded and saved to {filename}")
        return True
    except Exception as e:
        print(f"❌ An error occurred during recording: {e}")
        return False

def recognize_with_google(filename):
    """Recognizes speech using Google's online API."""
    print("\n--- ☁️ Recognizing with Google API ---")
    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(filename) as source:
            audio_data = recognizer.record(source)
        text = recognizer.recognize_google(audio_data)
        print(f"Speech recognized: '{text}'")
        print("✅ Speech successfully converted to text!")
    except sr.UnknownValueError:
        print("🤔 Speech Recognition could not understand audio. Please try speaking more clearly.")
    except sr.RequestError as e:
        print(f"📡 Could not request results from Google; {e}")

def recognize_with_whisper(filename):
    """Recognizes speech using OpenAI's Whisper model."""
    print("\n--- 🤖 Recognizing with Whisper (Offline) ---")
    try:
        model = whisper.load_model("base") # "base.en" is a good English-only option
        result = model.transcribe(filename)
        text = result["text"].strip()
        print(f"Speech recognized: '{text}'")
        print("✅ Speech successfully converted to text!")
    except Exception as e:
        print(f"❌ An error occurred with Whisper: {e}")
        
def recognize_with_vosk(filename):
    """Recognizes speech using the Vosk model."""
    print("\n--- 📦 Recognizing with Vosk (Offline) ---")
    if not os.path.exists(VOSK_MODEL_PATH):
        print(f"⚠️ Vosk model not found at '{VOSK_MODEL_PATH}'. Please check the path.")
        return
    try:
        model = vosk.Model(VOSK_MODEL_PATH)
        wf = wave.open(filename, "rb")
        recognizer = vosk.KaldiRecognizer(model, wf.getframerate())
        
        full_text = ""
        while True:
            data = wf.readframes(4000)
            if len(data) == 0: break
            if recognizer.AcceptWaveform(data):
                result = json.loads(recognizer.Result())
                full_text += result.get('text', '') + " "
        
        result = json.loads(recognizer.FinalResult())
        full_text += result.get('text', '')

        if full_text.strip():
            print(f"Speech recognized: '{full_text.strip()}'")
            print("✅ Speech successfully converted to text!")
        else:
            print("🤔 Vosk could not understand audio. Please try speaking more clearly.")
    except Exception as e:
        print(f"❌ An error occurred with Vosk: {e}")

print("✅ All functions defined.")

✅ All functions defined.


In [3]:
# Record 5 seconds of audio from your microphone
record_audio(RECORDING_FILENAME, duration=5)

🎤 Speak something for 5 seconds...
👍 Audio recorded and saved to output.wav


True

In [3]:
if os.path.exists(RECORDING_FILENAME):
    # Run all three recognition methods for comparison
    recognize_with_google(RECORDING_FILENAME)
    recognize_with_whisper(RECORDING_FILENAME)
    recognize_with_vosk(RECORDING_FILENAME)
else:
    print(f"Audio file '{RECORDING_FILENAME}' not found. Please run the recording cell first.")


--- ☁️ Recognizing with Google API ---
Speech recognized: 'hello hello'
✅ Speech successfully converted to text!

--- 🤖 Recognizing with Whisper (Offline) ---


C:\Users\shish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Speech recognized: 'Hello, hello, hello'
✅ Speech successfully converted to text!

--- 📦 Recognizing with Vosk (Offline) ---
Speech recognized: 'hello hello hello'
✅ Speech successfully converted to text!
